In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [2]:
from langchain_core.messages import AIMessage,HumanMessage
from pprint import pprint
messages=[AIMessage(content="Hello!"), HumanMessage(content="Hi there!")]
pprint(messages)

[AIMessage(content='Hello!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Hi there!', additional_kwargs={}, response_metadata={})]


In [ ]:
from langchain_groq import ChatGroq
llm_groq=ChatGroq(model="gpt-4")
llm_groq.invoke("Hey I am Ayushi")

def add(a: int, b: int) -> int:
    """
    Add a and b.

    Args:
        a (int): First integer
        b (int): Second integer

    Returns:
        int: Sum of a and b
    """
    return a + b

In [6]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list[AIMessage | HumanMessage], add_messages]

In [9]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

In [10]:
tools = [add]

In [11]:
print(tools)

[StructuredTool(name='add', description='Add two numbers.', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x000001A2ECFF9D00>)]


In [13]:
def llm_tool(state: State):
    return {
        "messages": [llm.invoke(state["messages"])]
    }

In [14]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

builder = StateGraph(State)

# Add Nodes
builder.add_node("llm_tool", llm_tool)
builder.add_node("tools", ToolNode(tools))

# Add Edge
builder.add_edge(START, "llm_tool")